In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile

with zipfile.ZipFile('/content/drive/MyDrive/IWMI/Data Science - Dataset.zip', 'r') as z:
    z.extractall('/content/dataset')

In [ ]:
import os
print(os.listdir('/content/dataset/data'))

['with_mask', 'without_mask']


In [1]:
# Importing required libraries
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Additional libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow import keras


class BasicPreprocessing:

    def __init__(self):
        self.img_size = 128
        self.dataset_path = '/content/dataset/data'
        self.images = []
        self.labels = []

    def import_dataset(self):
        for class_name in os.listdir(self.dataset_path):
            class_path = os.path.join(self.dataset_path, class_name)
            if not os.path.isdir(class_path):
                continue
            for img_file in os.listdir(class_path):
                img = cv2.imread(os.path.join(class_path, img_file))
                if img is None:
                    continue
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (self.img_size, self.img_size))
                self.images.append(img)
                self.labels.append(class_name)

    def normalize_images(self):
        X = np.array(self.images, dtype=np.float32) / 255.0
        le = LabelEncoder()
        y = le.fit_transform(self.labels)
        y = keras.utils.to_categorical(y, num_classes=2)
        return X, y, le

    def split_dataset(self, X, y):
        X_train, X_temp, y_train, y_temp = train_test_split(
            X, y, test_size=0.30, random_state=42, stratify=y
        )
        X_val, X_test, y_val, y_test = train_test_split(
            X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
        )
        return X_train, X_val, X_test, y_train, y_val, y_test

    def augment_data(self, X_train):
        datagen = tf.keras.preprocessing.image.ImageDataGenerator(
            rotation_range=15,
            width_shift_range=0.1,
            height_shift_range=0.1,
            horizontal_flip=True,
            zoom_range=0.1
        )
        datagen.fit(X_train)
        return datagen

    def visualize_samples(self, X_train, y_train, le):
        plt.figure(figsize=(10, 4))
        for i in range(6):
            plt.subplot(2, 3, i+1)
            plt.imshow(X_train[i])
            plt.title(le.classes_[np.argmax(y_train[i])], fontsize=9)
            plt.axis('off')
        plt.tight_layout()
        plt.show()


def main():

  print("IWMI Data Science Internship Assessment")

    prep = BasicPreprocessing()
    prep.import_dataset()
    X, y, le = prep.normalize_images()
    X_train, X_val, X_test, y_train, y_val, y_test = prep.split_dataset(X, y)
    datagen = prep.augment_data(X_train)
    prep.visualize_samples(X_train, y_train, le)

    return X_train, X_val, X_test, y_train, y_val, y_test, datagen, le


X_train, X_val, X_test, y_train, y_val, y_test, datagen, le = main()

FileNotFoundError: [Errno 2] No such file or directory: '/content/dataset/data'

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Dropout, Flatten, Dense
from tensorflow.keras.callbacks import ReduceLROnPlateau


class ModelDevelopment:

    def __init__(self):
        self.model = None
        self.img_size = 128

    def build_model(self):
        self.model = Sequential([
            Conv2D(32, (3, 3), activation='relu', input_shape=(self.img_size, self.img_size, 3)),
            BatchNormalization(),
            MaxPooling2D((2, 2)),

            Conv2D(64, (3, 3), activation='relu'),
            BatchNormalization(),
            MaxPooling2D((2, 2)),

            Conv2D(128, (3, 3), activation='relu'),
            BatchNormalization(),
            MaxPooling2D((2, 2)),

            Flatten(),
            Dense(128, activation='relu'),
            Dropout(0.5),
            Dense(2, activation='softmax')
        ])

        self.model.compile(
            optimizer='adam',
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )

        self.model.summary()
        return self.model

    def train_model(self, X_train, y_train, X_val, y_val, datagen):
        lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)

        self.history = self.model.fit(
            datagen.flow(X_train, y_train, batch_size=32),
            validation_data=(X_val, y_val),
            epochs=20,
            callbacks=[lr_scheduler]
        )
        return self.history

    def plot_training(self):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

        ax1.plot(self.history.history['accuracy'], label='Train')
        ax1.plot(self.history.history['val_accuracy'], label='Validation')
        ax1.set_title('Accuracy')
        ax1.legend()

        ax2.plot(self.history.history['loss'], label='Train')
        ax2.plot(self.history.history['val_loss'], label='Validation')
        ax2.set_title('Loss')
        ax2.legend()

        plt.tight_layout()
        plt.savefig('training_curves.png')
        plt.show()

    def save_model(self):
        self.model.save('best_model.h5')


md = ModelDevelopment()
md.build_model()
md.train_model(X_train, y_train, X_val, y_val, datagen)
md.plot_training()
md.save_model()

In [ ]:
import shutil
shutil.copy('training_curves.png', '/content/drive/MyDrive/IWMI/training_curves.png')
shutil.copy('best_model.h5', '/content/drive/MyDrive/IWMI/best_model.h5')

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

class BasicInference:

    def __init__(self, model):
        self.model = model

    def detect_images(self):
        face_cascade = cv2.CascadeClassifier(
            cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
        )

        test_img_path = '/content/dataset/data/with_mask/' + os.listdir('/content/dataset/data/with_mask/')[0]
        img = cv2.imread(test_img_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)

        for (x, y, w, h) in faces:
            face = img_rgb[y:y+h, x:x+w]
            face = cv2.resize(face, (128, 128))
            face = face / 255.0
            face = np.expand_dims(face, axis=0)

            pred = self.model.predict(face, verbose=0)
            label = le.classes_[np.argmax(pred)]
            confidence = np.max(pred) * 100

            cv2.rectangle(img_rgb, (x, y), (x+w, y+h), (0, 255, 0), 2)
            cv2.putText(img_rgb, f'{label} {confidence:.1f}%',
                       (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        plt.imshow(img_rgb)
        plt.axis('off')
        plt.show()

    def evaluate_model(self, X_test, y_test):
        y_pred = self.model.predict(X_test, verbose=0)
        y_pred_classes = np.argmax(y_pred, axis=1)
        y_true_classes = np.argmax(y_test, axis=1)

        print(classification_report(y_true_classes, y_pred_classes,
                                    target_names=le.classes_))

        cm = confusion_matrix(y_true_classes, y_pred_classes)
        plt.figure(figsize=(6, 4))
        sns.heatmap(cm, annot=True, fmt='d',
                    xticklabels=le.classes_, yticklabels=le.classes_)
        plt.ylabel('Actual')
        plt.xlabel('Predicted')
        plt.tight_layout()
        plt.savefig('confusion_matrix.png')
        plt.show()


bi = BasicInference(md.model)
bi.detect_images()
bi.evaluate_model(X_test, y_test)

In [ ]:
shutil.copy('confusion_matrix.png', '/content/drive/MyDrive/IWMI/confusion_matrix.png')